# 03_psychophysics — Session-1 hue-discrimination thresholds (JND) and identification (8AFC)

**Manuscript:** Results section 3; Methods 'Psychophysical tasks'; Supplementary S1 (tab:jnd_baseline, tab:staircase_pairs).

Interleaved staircases (two per hue pair) give the discrimination threshold t; gamma is the ratio to the control mean and z the deviation in control standard deviations (n = 7). `compute_hc_group_metrics.py` builds the control distribution, `a2_staircase_diagnosis.py` audits every staircase (censoring, track agreement) and `_staircase_pairs_table.py` prints tab:staircase_pairs from that audit. Trial-level behavioural files are not distributed; the notebook works from the per-participant summaries.

**How to read this notebook.** Every code cell loads committed result files from `results/` and compares the values it derives with the numbers printed in the manuscript (`V.check`). A check passes when the produced value equals the printed one at the printed precision, or satisfies the stated relation. Quantities that have no committed artifact are recorded as pointers (`V.flag`) rather than silently omitted. The last cell tallies the checks and writes `_checks_03_psychophysics.json`, which `run_notebooks.py` collects into `REPORT.md`.

Provenance: built by `tools/public_repo/build.py` of the development repository (commit 53c81c2); manuscript source in `../paper/`; check list in `../MANIFEST.md`; code map in `../MAP.md`.

**Source and code map**

| Result file | Producing script | What it holds |
|---|---|---|
| `results/hc_group_metrics.json` | `scripts/compute_hc_group_metrics.py` | control threshold mean / SD per pair and the deutan participant's session-1 thresholds |
| `results/sub-09_jnd_vs_hc.json / .csv` | `scripts/analyze_exp2_jnd_vs_hc.py` | protan session-1 thresholds ('nofilter') with z against the controls |
| `results/a2_staircase_diagnosis.json` | `scripts/a2_staircase_diagnosis.py` | every staircase: censoring at ceiling, high/low-start estimates and their spread (tab:staircase_pairs) |

In [1]:
import sys, json, csv
from pathlib import Path
sys.path.insert(0, str((Path.cwd() / ".." / "common").resolve()))
import numpy as np
from scipy import stats
import verify as V
from stats_helpers import crawford_howell, hedges_g, bh_fdr, wilson_interval
R = Path("results")
def J(name):
    with open(R / name) as f:
        return json.load(f)
HC = [f"sub-{i:02d}" for i in range(1, 8)]
CVD = {"deutan": "sub-08", "protan": "sub-09"}
ROIS = ["V1", "V2", "V3", "hV4"]
HUES = ["red", "orange", "yellow", "green", "cyan", "blue", "purple", "magenta"]

hg = J("hc_group_metrics.json")
z8 = J("sub-08_jnd_vs_hc.json"); z9 = J("sub-09_jnd_vs_hc.json")
def csv_rows(name):
    with open(R / name) as f:
        return {row["pair_name"]: row for row in csv.DictReader(f)}
c8 = csv_rows("sub-08_jnd_vs_hc.csv"); c9 = csv_rows("sub-09_jnd_vs_hc.csv")
a2 = J("a2_staircase_diagnosis.json")
PAIRS = ["orange-yellow", "yellow-green", "green-blue", "cyan-magenta", "yellow-purple", "blue-purple", "red-orange", "red-cyan"]

V.start("03_psychophysics")

### Session-1 thresholds (Results section 3; Supplementary tab:jnd_baseline)
t = staircase level, gamma = t / control mean, z = (t - control mean) / control SD (n = 7).

| id | manuscript | quantity | reported |
|---|---|---|---|
| 03.01 | tab:jnd_baseline | 8 pairs x (control mean, SD, deutan t/gamma/z, protan t/gamma/z) | `64 cells, see 03.T1.*` |
| 03.02 | Results §3 ¶2 | mean |z| over eight pairs, deutan | `2.24` |
| 03.03 | Results §3 ¶2 | mean |z|, protan | `0.9` |
| 03.04 | Results §3 ¶2 | remaining pairs lie within |z| <= 1.23 in both participants | `1.23` |
| 03.05 | Results §3 ¶1 | deutan exceeds z = +4 on both deutan-axis pairs and on yellow-purple | `4.0` |
| 03.06 | Results §3 ¶1 | protan exceeds z = +2 on green-blue only | `['green-blue']` |
| 03.07 | tab:jnd_baseline caption | staircases collected in the study | `208` |
| 03.08 | tab:jnd_baseline caption | staircases censored at the largest presentable separation | `2` |
| 03.09 | tab:jnd_baseline caption | both censored staircases are the deutan orange-yellow pair | `[('sub-08_jnd_ses1_no_filter', 'orange-yellow')]` |

In [2]:
T = {"orange-yellow": (0.278, 0.135, 0.840, 3.02, 4.15, 0.193, 0.69, -0.63), "yellow-green": (0.090, 0.044, 0.278, 3.10, 4.32, 0.128, 1.42, 0.87),
     "green-blue": (0.079, 0.032, 0.078, 0.98, -0.06, 0.155, 1.95, 2.36), "cyan-magenta": (0.042, 0.017, 0.040, 0.95, -0.13, 0.038, 0.89, -0.28),
     "yellow-purple": (0.022, 0.006, 0.063, 2.87, 6.70, 0.028, 1.26, 0.94), "blue-purple": (0.164, 0.093, 0.120, 0.73, -0.48, 0.148, 0.90, -0.18),
     "red-orange": (0.124, 0.072, 0.063, 0.50, -0.85, 0.075, 0.61, -0.68), "red-cyan": (0.034, 0.015, 0.015, 0.45, -1.23, 0.015, 0.45, -1.23)}
zs = {"deutan": {}, "protan": {}}
for pair, (m_r, sd_r, t8_r, g8_r, z8_r, t9_r, g9_r, z9_r) in T.items():
    m, sd = hg[pair]["hc_mean"], hg[pair]["hc_std"]
    t8 = float(c8[pair]["nofilter"]); t9 = float(c9[pair]["nofilter"])
    zs["deutan"][pair] = (t8 - m) / sd; zs["protan"][pair] = (t9 - m) / sd
    V.check(f"03.T1.{pair}.mean", f"tab:jnd_baseline {pair} control mean", m, m_r, nd=3)
    V.check(f"03.T1.{pair}.sd", f"tab:jnd_baseline {pair} control SD", sd, sd_r, nd=3)
    V.check(f"03.T1.{pair}.deutan_t", f"tab:jnd_baseline {pair} deutan t", t8, t8_r, nd=3)
    V.check(f"03.T1.{pair}.deutan_gamma", f"tab:jnd_baseline {pair} deutan gamma", t8 / m, g8_r, nd=2)
    V.check(f"03.T1.{pair}.deutan_z", f"tab:jnd_baseline {pair} deutan z", (t8 - m) / sd, z8_r, nd=2)
    V.check(f"03.T1.{pair}.protan_t", f"tab:jnd_baseline {pair} protan t", t9, t9_r, nd=3)
    V.check(f"03.T1.{pair}.protan_gamma", f"tab:jnd_baseline {pair} protan gamma", t9 / m, g9_r, nd=2)
    V.check(f"03.T1.{pair}.protan_z", f"tab:jnd_baseline {pair} protan z", (t9 - m) / sd, z9_r, nd=2)
mean_abs_z = {k: np.mean([abs(v) for v in d.values()]) for k, d in zs.items()}
elev = {"deutan": ["orange-yellow", "yellow-green", "yellow-purple"], "protan": ["green-blue"]}
rest_max = max(abs(zs[k][p]) for k in zs for p in PAIRS if p not in elev[k])
census = a2["census"]
print(mean_abs_z, rest_max, census["n_staircases"], census["n_censored"])
V.table('03.01', 'tab:jnd_baseline | 8 pairs x (control mean, SD, deutan t/gamma/z, protan t/gamma/z)', '64 cells, see 03.T1.*')
V.check('03.02', 'Results §3 ¶2 | mean |z| over eight pairs, deutan', mean_abs_z["deutan"], 2.24, nd=2)
V.check('03.03', 'Results §3 ¶2 | mean |z|, protan', mean_abs_z["protan"], 0.9, nd=2)
V.check('03.04', 'Results §3 ¶2 | remaining pairs lie within |z| <= 1.23 in both participants', rest_max, 1.23, nd=2)
V.check('03.05', 'Results §3 ¶1 | deutan exceeds z = +4 on both deutan-axis pairs and on yellow-purple', min(zs["deutan"][p] for p in elev["deutan"]), 4.0, mode='gt')
V.check('03.06', 'Results §3 ¶1 | protan exceeds z = +2 on green-blue only', [p for p in PAIRS if zs["protan"][p] > 2], ['green-blue'], mode='eq')
V.check('03.07', 'tab:jnd_baseline caption | staircases collected in the study', census["n_staircases"], 208, mode='eq')
V.check('03.08', 'tab:jnd_baseline caption | staircases censored at the largest presentable separation', census["n_censored"], 2, mode='eq')
V.check('03.09', 'tab:jnd_baseline caption | both censored staircases are the deutan orange-yellow pair', sorted({(c["source"], c["pair"]) for c in census["censored"]}), [('sub-08_jnd_ses1_no_filter', 'orange-yellow')], mode='eq')

[OK ] 03.T1.orange-yellow.mean tab:jnd_baseline orange-yellow control mean: produced=0.2781  reported=0.278
[OK ] 03.T1.orange-yellow.sd tab:jnd_baseline orange-yellow control SD: produced=0.1353  reported=0.135
[OK ] 03.T1.orange-yellow.deutan_t tab:jnd_baseline orange-yellow deutan t: produced=0.84  reported=0.84
[OK ] 03.T1.orange-yellow.deutan_gamma tab:jnd_baseline orange-yellow deutan gamma: produced=3.021  reported=3.02
[OK ] 03.T1.orange-yellow.deutan_z tab:jnd_baseline orange-yellow deutan z: produced=4.154  reported=4.15
[OK ] 03.T1.orange-yellow.protan_t tab:jnd_baseline orange-yellow protan t: produced=0.1925  reported=0.193
[OK ] 03.T1.orange-yellow.protan_gamma tab:jnd_baseline orange-yellow protan gamma: produced=0.6922  reported=0.69
[OK ] 03.T1.orange-yellow.protan_z tab:jnd_baseline orange-yellow protan z: produced=-0.6327  reported=-0.63
[OK ] 03.T1.yellow-green.mean tab:jnd_baseline yellow-green control mean: produced=0.08952  reported=0.09
[~~ ] 03.T1.yellow-green.

### Both staircase estimates per pair (Supplementary tab:staircase_pairs)
High-start / low-start estimates for two participants x three conditions x eight pairs; thresholds elsewhere are the mean of the two.

| id | manuscript | quantity | reported |
|---|---|---|---|
| 03.10 | tab:staircase_pairs | 8 pairs x 6 columns x (high, low) | `96 cells, see 03.T2.*` |
| 03.11 | S1 ¶ after tab:staircase_pairs | number of pair-level cells (13 files x 8 pairs) | `104` |
| 03.12 | S1 | median difference between the two staircases | `0.015` |
| 03.13 | S1 | cells whose two estimates differ by more than 0.10 | `3` |
| 03.14 | S1 | largest difference | `0.515` |
| 03.15 | S1 | it is the protan participant's orange-yellow pair under the individualized filter | `('sub-09/individualized', 'orange-yellow')` |
| 03.16 | S1 | that cell's z with the mean of the two tracks | `1.33` |
| 03.17 | S1 | z restricted to the converged track | `-0.58` |
| 03.18 | S1 | protan mean |z| under the individualized filter | `0.93` |
| 03.19 | S1 | same with the converged track | `0.84` |

In [3]:
T_SP = {
 "red-orange":    ("0.060/0.065", "0.160/0.210", "0.140/0.130", "0.075/0.075", "0.110/0.145", "0.070/0.055"),
 "orange-yellow": ("0.870/0.810", "0.080/0.110", "0.060/0.032", "0.175/0.210", "0.170/0.185", "0.715/0.200"),
 "yellow-green":  ("0.280/0.275", "0.090/0.075", "0.145/0.165", "0.125/0.130", "0.057/0.050", "0.055/0.065"),
 "green-blue":    ("0.065/0.090", "0.150/0.110", "0.045/0.060", "0.145/0.165", "0.260/0.225", "0.135/0.080"),
 "blue-purple":   ("0.115/0.125", "0.050/0.095", "0.110/0.200", "0.125/0.170", "0.135/0.110", "0.150/0.105"),
 "yellow-purple": ("0.060/0.065", "0.015/0.020", "0.035/0.025", "0.020/0.035", "0.020/0.015", "0.015/0.015"),
 "red-cyan":      ("0.015/0.015", "0.040/0.018", "0.015/0.045", "0.015/0.015", "0.030/0.040", "0.020/0.025"),
 "cyan-magenta":  ("0.035/0.045", "0.030/0.025", "0.040/0.033", "0.040/0.035", "0.150/0.140", "0.015/0.020"),
}
COLS = [("sub-08", "session1"), ("sub-08", "deployed"), ("sub-08", "individualized"), ("sub-09", "session1"), ("sub-09", "deployed"), ("sub-09", "individualized")]
spreads = []
for pair, cells in T_SP.items():
    for (sub, cond), rep in zip(COLS, cells):
        p = a2["conditions"][f"{sub}/{cond}"]["pairs"][pair]
        hi_r, lo_r = (float(x) for x in rep.split("/"))
        V.check(f"03.T2.{pair}.{sub}.{cond}.hi", f"tab:staircase_pairs {pair} {sub} {cond} high-start", p["sc_hi"], hi_r, nd=3)
        V.check(f"03.T2.{pair}.{sub}.{cond}.lo", f"tab:staircase_pairs {pair} {sub} {cond} low-start", p["sc_lo"], lo_r, nd=3)
all_spreads = [p["spread"] for cond in a2["conditions"].values() for p in cond["pairs"].values()]
worst = max(((c, pair, p["spread"]) for c, cond in a2["conditions"].items() for pair, p in cond["pairs"].items()), key=lambda x: x[2])
# converged-track counterfactual for the protan individualized orange-yellow cell
m, sd = hg["orange-yellow"]["hc_mean"], hg["orange-yellow"]["hc_std"]
z_mean_track = (a2["conditions"]["sub-09/individualized"]["pairs"]["orange-yellow"]["threshold"] - m) / sd
z_low_track = (a2["conditions"]["sub-09/individualized"]["pairs"]["orange-yellow"]["sc_lo"] - m) / sd
zopt = {pair: float(c9[pair]["z_optimal"]) for pair in PAIRS}
mean_abs_z_opt = np.mean([abs(v) for v in zopt.values()])
mean_abs_z_opt_cf = np.mean([abs(z_low_track if pair == "orange-yellow" else zopt[pair]) for pair in PAIRS])
print(len(all_spreads), np.median(all_spreads), sum(s > 0.10 for s in all_spreads), worst, z_mean_track, z_low_track, mean_abs_z_opt, mean_abs_z_opt_cf)
V.table('03.10', 'tab:staircase_pairs | 8 pairs x 6 columns x (high, low)', '96 cells, see 03.T2.*')
V.check('03.11', 'S1 ¶ after tab:staircase_pairs | number of pair-level cells (13 files x 8 pairs)', len(all_spreads), 104, mode='eq')
V.check('03.12', 'S1 | median difference between the two staircases', float(np.median(all_spreads)), 0.015, nd=3)
V.check('03.13', 'S1 | cells whose two estimates differ by more than 0.10', sum(s > 0.10 for s in all_spreads), 3, mode='eq')
V.check('03.14', 'S1 | largest difference', worst[2], 0.515, nd=3)
V.check('03.15', "S1 | it is the protan participant's orange-yellow pair under the individualized filter", (worst[0], worst[1]), ('sub-09/individualized', 'orange-yellow'), mode='eq')
V.check('03.16', "S1 | that cell's z with the mean of the two tracks", z_mean_track, 1.33, nd=2)
V.check('03.17', 'S1 | z restricted to the converged track', z_low_track, -0.58, nd=2)
V.check('03.18', 'S1 | protan mean |z| under the individualized filter', mean_abs_z_opt, 0.93, nd=2)
V.check('03.19', 'S1 | same with the converged track', mean_abs_z_opt_cf, 0.84, nd=2)

[OK ] 03.T2.red-orange.sub-08.session1.hi tab:staircase_pairs red-orange sub-08 session1 high-start: produced=0.06  reported=0.06
[OK ] 03.T2.red-orange.sub-08.session1.lo tab:staircase_pairs red-orange sub-08 session1 low-start: produced=0.065  reported=0.065
[OK ] 03.T2.red-orange.sub-08.deployed.hi tab:staircase_pairs red-orange sub-08 deployed high-start: produced=0.16  reported=0.16
[OK ] 03.T2.red-orange.sub-08.deployed.lo tab:staircase_pairs red-orange sub-08 deployed low-start: produced=0.21  reported=0.21
[OK ] 03.T2.red-orange.sub-08.individualized.hi tab:staircase_pairs red-orange sub-08 individualized high-start: produced=0.14  reported=0.14
[OK ] 03.T2.red-orange.sub-08.individualized.lo tab:staircase_pairs red-orange sub-08 individualized low-start: produced=0.13  reported=0.13
[OK ] 03.T2.red-orange.sub-09.session1.hi tab:staircase_pairs red-orange sub-09 session1 high-start: produced=0.075  reported=0.075
[OK ] 03.T2.red-orange.sub-09.session1.lo tab:staircase_pairs red

In [4]:
V.summary()


=== 03_psychophysics: 175/177 numeric checks reproduced exactly; 2 within one unit of the last printed digit; 0 mismatch, 0 error, 0 pointer-only ===
  NEAR     03.T1.yellow-green.sd tab:jnd_baseline yellow-green control SD: produced=0.0435 reported=0.044
  NEAR     03.T1.cyan-magenta.sd tab:jnd_baseline cyan-magenta control SD: produced=0.01648 reported=0.017
